### 0. Imports and Paths

In [1]:
from pathlib import Path
import cv2
import numpy as np
import shutil
from tqdm import tqdm

In [2]:
# ROOT of your YOLO dataset
ROOT = Path("../data_for_yolo")

IMG_TEST = ROOT / "images" / "test"
LBL_TEST = ROOT / "labels" / "test"

In [3]:
assert IMG_TEST.exists(), "images/test does not exist"
assert LBL_TEST.exists(), "labels/test does not exist"

### 1. Perturbation Functions

In [4]:
def apply_blur(img, k):
    if k % 2 == 0:
        k += 1
    return cv2.GaussianBlur(img, (k, k), 0)

In [5]:
def apply_lighting(img, alpha, beta):
    return cv2.convertScaleAbs(img, alpha=alpha, beta=beta)

In [6]:
def apply_occlusion(img, labels, area_ratio):
    h, w, _ = img.shape
    img_area = h * w

    occ_area = max(1, int(img_area * area_ratio))

    # Compute occlusion size
    occ_w = int(np.sqrt(occ_area))
    occ_h = int(occ_area / occ_w)

    # --- CRITICAL SAFETY CLAMPS ---
    occ_w = np.clip(occ_w, 1, w - 1)
    occ_h = np.clip(occ_h, 1, h - 1)

    # Convert YOLO labels to pixel boxes
    boxes = []
    if labels is not None:
        for _, xc, yc, bw, bh in labels:
            x1 = int((xc - bw / 2) * w)
            y1 = int((yc - bh / 2) * h)
            x2 = int((xc + bw / 2) * w)
            y2 = int((yc + bh / 2) * h)
            boxes.append((x1, y1, x2, y2))

    # Try to place occlusion without fully covering any GT box
    for _ in range(50):
        x = np.random.randint(0, w - occ_w)
        y = np.random.randint(0, h - occ_h)
        occ_box = (x, y, x + occ_w, y + occ_h)

        fully_blocks = False
        for bx1, by1, bx2, by2 in boxes:
            if (occ_box[0] <= bx1 and occ_box[1] <= by1 and
                occ_box[2] >= bx2 and occ_box[3] >= by2):
                fully_blocks = True
                break

        if not fully_blocks:
            img = img.copy()
            img[y:y+occ_h, x:x+occ_w] = 0
            return img

    # Fallback
    img = img.copy()
    img[y:y+occ_h, x:x+occ_w] = 0
    return img

### 2. Load YOLO Labels

In [7]:
def load_labels(label_path):
    labels = []
    with open(label_path) as f:
        for line in f:
            cls, x, y, w, h = map(float, line.strip().split())
            labels.append((cls, x, y, w, h))
    return labels

### 3. Core Generator Function

In [8]:
def generate_variant(transform_fn, img_dst, lbl_dst, needs_labels=False):
    for img_path in tqdm(list(IMG_TEST.glob("*"))):
        if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
            continue

        img = cv2.imread(str(img_path))

        label_path = LBL_TEST / (img_path.stem + ".txt")
        labels = load_labels(label_path) if (needs_labels and label_path.exists()) else None

        img = transform_fn(img, labels) if needs_labels else transform_fn(img)

        cv2.imwrite(str(img_dst / img_path.name), img)

        if label_path.exists():
            shutil.copy(label_path, lbl_dst / label_path.name)

### 4. Generate BLUR Sets

In [20]:
blur_levels = {
    "test_blur_s1": 31,
    "test_blur_s2": 71,
    "test_blur_s3": 151,
}

for name, k in blur_levels.items():
    generate_variant(
        lambda img, k=k: apply_blur(img, k),
        ROOT / "images" / name,
        ROOT / "labels" / name,
    )

100%|██████████| 1620/1620 [1:16:01<00:00,  2.82s/it]    


### 5. Gererate LIGHTING Sets

In [21]:
light_levels = {
    "test_light_s1": (0.9, -15),
    "test_light_s2": (0.75, -30),
    "test_light_s3": (0.6, -50),
}

for name, (a, b) in light_levels.items():
    generate_variant(
        lambda img, a=a, b=b: apply_lighting(img, a, b),
        ROOT / "images" / name,
        ROOT / "labels" / name,
    )

100%|██████████| 1620/1620 [06:04<00:00,  4.45it/s]


### 6. Generate OCCLUSION Sets

In [9]:
occ_levels = {
    "test_occ_s1": 0.10,
    "test_occ_s2": 0.20,
    "test_occ_s3": 0.35,
}

for name, ratio in occ_levels.items():
    generate_variant(
        lambda img, labels, r=ratio: apply_occlusion(img, labels, r),
        ROOT / "images" / name,
        ROOT / "labels" / name,
        needs_labels=True
    )

100%|██████████| 1620/1620 [10:26<00:00,  2.59it/s] 


### 7. Sanity Check

In [10]:
for name in sorted([p.name for p in (ROOT / "images").iterdir() if p.name.startswith("test_")]):
    imgs = len(list((ROOT / "images" / name).glob("*")))
    lbls = len(list((ROOT / "labels" / name).glob("*.txt")))
    print(f"{name}: {imgs} images | {lbls} labels")

test_blur_s1: 1620 images | 1620 labels
test_blur_s2: 1620 images | 1620 labels
test_blur_s3: 1620 images | 1620 labels
test_light_s1: 1620 images | 1620 labels
test_light_s2: 1620 images | 1620 labels
test_light_s3: 1620 images | 1620 labels
test_occ_s1: 1620 images | 1620 labels
test_occ_s2: 1620 images | 1620 labels
test_occ_s3: 1620 images | 1620 labels
